In [1]:
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def goal_test(self, state):
        if isinstance(self.goal, list):
            return state in self.goal
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        return c + 1

In [4]:
# Exercise 3.1

class EightPuzzle(Problem):
    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        super().__init__(initial, goal)

    def actions(self, state):
        # Actions define the movement of the blank space (Up, Down, Left, Right) [cite: 51]
        moves = []
        idx = state.index(0)
        if idx not in (0, 1, 2): moves.append('UP')
        if idx not in (6, 7, 8): moves.append('DOWN')
        if idx not in (0, 3, 6): moves.append('LEFT')
        if idx not in (2, 5, 8): moves.append('RIGHT')
        return moves

    def result(self, state, action):
        # The transition model returns the resulting state after sliding [cite: 51]
        new_state = list(state)
        idx = state.index(0)
        if action == 'UP': neighbor = idx - 3
        elif action == 'DOWN': neighbor = idx + 3
        elif action == 'LEFT': neighbor = idx - 1
        elif action == 'RIGHT': neighbor = idx + 1
        
        new_state[idx], new_state[neighbor] = new_state[neighbor], new_state[idx]
        return tuple(new_state)

In [6]:
# BFS - Algorithm for Execution

from collections import deque

def breadth_first_search(problem):
    """Finds the shallowest node in the search tree."""
    node = problem.initial
    if problem.goal_test(node):
        return [node]
    
    # Frontier is a FIFO queue for BFS 
    frontier = deque([[node]])
    explored = set()
    
    while frontier:
        path = frontier.popleft()
        current_state = path[-1]
        
        if current_state not in explored:
            explored.add(current_state)
            
            for action in problem.actions(current_state):
                child = problem.result(current_state, action)
                new_path = list(path)
                new_path.append(child)
                
                if problem.goal_test(child):
                    return new_path
                
                frontier.append(new_path)
    return None

In [7]:
# for 3.1 
start_state = (2, 1, 3, 4, 7, 6, 5, 8, 0) 
puzzle = EightPuzzle(initial=start_state)
solution = breadth_first_search(puzzle)

print("\n--- 8-Puzzle Solution ---")
for step, state in enumerate(solution):
    print(f"Step {step}: {state[0:3]}\n        {state[3:6]}\n        {state[6:9]}\n")


--- 8-Puzzle Solution ---
Step 0: (2, 1, 3)
        (4, 7, 6)
        (5, 8, 0)

Step 1: (2, 1, 3)
        (4, 7, 0)
        (5, 8, 6)

Step 2: (2, 1, 0)
        (4, 7, 3)
        (5, 8, 6)

Step 3: (2, 0, 1)
        (4, 7, 3)
        (5, 8, 6)

Step 4: (0, 2, 1)
        (4, 7, 3)
        (5, 8, 6)

Step 5: (4, 2, 1)
        (0, 7, 3)
        (5, 8, 6)

Step 6: (4, 2, 1)
        (7, 0, 3)
        (5, 8, 6)

Step 7: (4, 0, 1)
        (7, 2, 3)
        (5, 8, 6)

Step 8: (4, 1, 0)
        (7, 2, 3)
        (5, 8, 6)

Step 9: (4, 1, 3)
        (7, 2, 0)
        (5, 8, 6)

Step 10: (4, 1, 3)
        (7, 2, 6)
        (5, 8, 0)

Step 11: (4, 1, 3)
        (7, 2, 6)
        (5, 0, 8)

Step 12: (4, 1, 3)
        (7, 2, 6)
        (0, 5, 8)

Step 13: (4, 1, 3)
        (0, 2, 6)
        (7, 5, 8)

Step 14: (0, 1, 3)
        (4, 2, 6)
        (7, 5, 8)

Step 15: (1, 0, 3)
        (4, 2, 6)
        (7, 5, 8)

Step 16: (1, 2, 3)
        (4, 0, 6)
        (7, 5, 8)

Step 17: (1, 2, 3)
        (4, 

In [3]:
# Exercise 3.3

class MissionariesAndCannibals(Problem):
    def __init__(self):
        # State: (Missionaries, Cannibals, Boat_at_Start_Bank) 
        super().__init__(initial=(3, 3, 1), goal=(0, 0, 0))

    def actions(self, state):
        m, c, b = state
        possible_moves = [(1,0), (2,0), (0,1), (0,2), (1,1)]
        valid = []
        for dm, dc in possible_moves:
            # Transition based on boat position 
            new_s = (m-dm, c-dc, 0) if b == 1 else (m+dm, c+dc, 1)
            if self.is_safe(new_s):
                valid.append((dm, dc))
        return valid

    def is_safe(self, state):
        m, c, b = state
        if not (0 <= m <= 3 and 0 <= c <= 3): return False
        if m > 0 and m < c: return False # Start bank safety
        if (3-m) > 0 and (3-m) < (3-c): return False # Dest bank safety
        return True

    def result(self, state, action):
        m, c, b = state
        dm, dc = action
        return (m-dm, c-dc, 0) if b == 1 else (m+dm, c+dc, 1)

In [8]:
# for 3.3

mc_problem = MissionariesAndCannibals()
solution = breadth_first_search(mc_problem)

print("--- Missionaries and Cannibals Solution ---")
for step, state in enumerate(solution):
    print(f"Step {step}: {state}")

--- Missionaries and Cannibals Solution ---
Step 0: (3, 3, 1)
Step 1: (3, 1, 0)
Step 2: (3, 2, 1)
Step 3: (3, 0, 0)
Step 4: (3, 1, 1)
Step 5: (1, 1, 0)
Step 6: (2, 2, 1)
Step 7: (0, 2, 0)
Step 8: (0, 3, 1)
Step 9: (0, 1, 0)
Step 10: (1, 1, 1)
Step 11: (0, 0, 0)
